# DAG Coupling & Retry Safety

Tightly coupled DAG tasks that share mutable state create cascading failures:
a retry after a partial failure corrupts data, and a "mega-task" that bundles
everything wastes hours of compute on re-runs. This notebook simulates both
anti-patterns using plain Python (no Airflow required) and shows how idempotent
tasks with atomic boundaries make retries safe.

In [ ]:
!pip install pandas numpy --quiet

## Simulate a Data Pipeline DAG

We model a 5-step pipeline as Python functions with a shared dict-based "data store"
(simulating a database or filesystem):

```
extract → clean → enrich → aggregate → publish
```

Each task reads from and writes to the store. We inject a failure in `aggregate()`
to simulate a real-world crash.

In [ ]:
import pandas as pd
import numpy as np
import copy
import time

np.random.seed(42)

# Shared mutable state (simulates a database/filesystem)
store = {}


def extract(store):
    """Simulate reading raw data from a source system."""
    time.sleep(0.1)  # simulate I/O
    raw = pd.DataFrame({
        "user_id": np.random.randint(1, 100, size=1000),
        "amount": np.round(np.random.uniform(1, 500, size=1000), 2),
        "category": np.random.choice(["A", "B", "C", None], size=1000),
    })
    store["raw"] = raw
    print(f"  extract: {len(raw)} rows")


def clean(store):
    """Drop nulls, fix dtypes."""
    time.sleep(0.1)
    cleaned = store["raw"].dropna().copy()
    store["cleaned"] = cleaned
    print(f"  clean: {len(cleaned)} rows")


def enrich_bad(store):
    """ANTI-PATTERN: appends to existing data instead of overwriting."""
    time.sleep(0.1)
    enriched = store["cleaned"].copy()
    enriched["score"] = enriched["amount"] * np.random.uniform(0.8, 1.2, size=len(enriched))

    # BUG: appends instead of overwrites — not idempotent!
    if "enriched" in store:
        store["enriched"] = pd.concat([store["enriched"], enriched], ignore_index=True)
    else:
        store["enriched"] = enriched
    print(f"  enrich: store['enriched'] now has {len(store['enriched'])} rows")


def aggregate(store, fail=False):
    """Aggregate by category."""
    time.sleep(0.1)
    if fail:
        print("  aggregate: CRASH! (simulated OOM)")
        raise MemoryError("Simulated OOM in aggregate")
    agg = store["enriched"].groupby("category").agg(
        total_amount=("amount", "sum"),
        avg_score=("score", "mean"),
        count=("user_id", "count"),
    ).reset_index()
    store["aggregated"] = agg
    print(f"  aggregate: {len(agg)} categories")


def publish(store):
    """Write final report."""
    time.sleep(0.1)
    store["published"] = store["aggregated"].copy()
    print(f"  publish: done")

## Anti-Pattern 1: Shared Mutable State + Non-Idempotent Tasks

The `enrich_bad` function **appends** to existing state instead of overwriting it.
On a successful first run this looks fine. But when `aggregate` crashes and we retry
from `enrich`, the enriched data gets doubled.

In [ ]:
# First run — aggregate crashes
store = {}
print("=== Run 1 (aggregate will crash) ===")
extract(store)
clean(store)
enrich_bad(store)
try:
    aggregate(store, fail=True)
except MemoryError:
    pass

enriched_after_run1 = len(store["enriched"])
print(f"\nEnriched rows after run 1: {enriched_after_run1}")

# Retry from enrich (the last successful step needs re-run)
print("\n=== Retry from enrich ===")
enrich_bad(store)  # BUG: this appends instead of overwriting!
aggregate(store, fail=False)

enriched_after_retry = len(store["enriched"])
print(f"\nEnriched rows after retry: {enriched_after_retry}")
print(f"Expected: {enriched_after_run1}, Got: {enriched_after_retry} — DATA DOUBLED!")
print("\nThe aggregate results are now computed on 2x the data — silently wrong.")
display(store["aggregated"])

## Anti-Pattern 2: The Mega-Task

An alternative anti-pattern: bundle all steps into a single function. If it fails
at step 4, you re-run the entire pipeline from scratch — wasting all the work from
steps 1–3.

In [ ]:
def mega_task(fail_at_aggregate=False):
    """Everything in one function — no checkpointing."""
    s = {}
    start = time.time()

    # Step 1: extract (slow)
    time.sleep(0.3)
    s["raw"] = pd.DataFrame({
        "user_id": np.random.randint(1, 100, size=1000),
        "amount": np.round(np.random.uniform(1, 500, size=1000), 2),
        "category": np.random.choice(["A", "B", "C", None], size=1000),
    })

    # Step 2: clean (slow)
    time.sleep(0.3)
    s["cleaned"] = s["raw"].dropna()

    # Step 3: enrich (slow)
    time.sleep(0.3)
    enriched = s["cleaned"].copy()
    enriched["score"] = enriched["amount"] * np.random.uniform(0.8, 1.2, size=len(enriched))
    s["enriched"] = enriched

    # Step 4: aggregate (may crash)
    if fail_at_aggregate:
        elapsed = time.time() - start
        print(f"  mega_task: CRASHED at step 4 after {elapsed:.1f}s")
        print(f"  All work from steps 1-3 is LOST — must restart from scratch.")
        raise MemoryError("Simulated OOM")

    time.sleep(0.1)
    s["aggregated"] = s["enriched"].groupby("category").agg(
        total_amount=("amount", "sum"), count=("user_id", "count")
    ).reset_index()

    elapsed = time.time() - start
    print(f"  mega_task: completed in {elapsed:.1f}s")
    return s


print("=== Mega-task (fails at step 4) ===")
try:
    mega_task(fail_at_aggregate=True)
except MemoryError:
    pass

print("\n=== Mega-task (full retry from scratch) ===")
result = mega_task(fail_at_aggregate=False)
print("\nTotal time for recovery: ~2x the original run. With real data this could be hours.")

## Correct Pattern: Idempotent Tasks with Atomic Boundaries

Each task:
1. **Overwrites** its output (never appends) — makes it idempotent.
2. Writes to a **versioned checkpoint** — enables safe retry from any point.
3. Uses **write-to-temp-then-swap** — if the task crashes mid-write, the previous
   checkpoint remains intact.

In [ ]:
class CheckpointedStore:
    """A store that supports versioned, atomic writes."""

    def __init__(self):
        self._data = {}       # key -> value (committed)
        self._versions = {}   # key -> version counter

    def read(self, key):
        if key not in self._data:
            raise KeyError(f"No checkpoint found for '{key}'")
        return self._data[key].copy()  # return a copy for safety

    def write(self, key, value):
        """Atomic overwrite — always replaces, never appends."""
        # In production: write to temp path, then rename (atomic on most filesystems)
        self._data[key] = value.copy()
        self._versions[key] = self._versions.get(key, 0) + 1

    def version(self, key):
        return self._versions.get(key, 0)

    def status(self):
        for key in self._data:
            print(f"  {key}: {len(self._data[key])} rows, version {self._versions[key]}")


def extract_safe(store):
    raw = pd.DataFrame({
        "user_id": np.random.randint(1, 100, size=1000),
        "amount": np.round(np.random.uniform(1, 500, size=1000), 2),
        "category": np.random.choice(["A", "B", "C", None], size=1000),
    })
    store.write("raw", raw)  # overwrite, not append
    print(f"  extract: {len(raw)} rows")


def clean_safe(store):
    raw = store.read("raw")
    cleaned = raw.dropna()
    store.write("cleaned", cleaned)  # overwrite
    print(f"  clean: {len(cleaned)} rows")


def enrich_safe(store):
    cleaned = store.read("cleaned")
    enriched = cleaned.copy()
    enriched["score"] = enriched["amount"] * np.random.uniform(0.8, 1.2, size=len(enriched))
    store.write("enriched", enriched)  # overwrite — idempotent!
    print(f"  enrich: {len(enriched)} rows")


def aggregate_safe(store, fail=False):
    if fail:
        print("  aggregate: CRASH!")
        raise MemoryError("Simulated OOM")
    enriched = store.read("enriched")
    agg = enriched.groupby("category").agg(
        total_amount=("amount", "sum"),
        avg_score=("score", "mean"),
        count=("user_id", "count"),
    ).reset_index()
    store.write("aggregated", agg)
    print(f"  aggregate: {len(agg)} categories")


def publish_safe(store):
    agg = store.read("aggregated")
    store.write("published", agg)
    print(f"  publish: done")

In [ ]:
# Run with crash at aggregate
safe_store = CheckpointedStore()
np.random.seed(42)  # reset for reproducibility

print("=== Run 1 (aggregate will crash) ===")
extract_safe(safe_store)
clean_safe(safe_store)
enrich_safe(safe_store)
try:
    aggregate_safe(safe_store, fail=True)
except MemoryError:
    pass

print("\nStore status after crash:")
safe_store.status()
enriched_v1 = len(safe_store.read("enriched"))

# Retry from enrich (safe because overwrite, not append)
print("\n=== Retry from enrich ===")
enrich_safe(safe_store)
aggregate_safe(safe_store, fail=False)
publish_safe(safe_store)

enriched_v2 = len(safe_store.read("enriched"))
print(f"\nEnriched rows: run 1 = {enriched_v1}, retry = {enriched_v2}")
print(f"Idempotent? {enriched_v1 == enriched_v2}")

print("\nFinal store status:")
safe_store.status()

print("\nFinal result:")
display(safe_store.read("published"))

In [ ]:
# Run the entire pipeline twice to prove full idempotency
np.random.seed(42)
store_a = CheckpointedStore()
extract_safe(store_a)
clean_safe(store_a)
enrich_safe(store_a)
aggregate_safe(store_a)
publish_safe(store_a)
result_a = store_a.read("published")

np.random.seed(42)
store_b = CheckpointedStore()
extract_safe(store_b)
clean_safe(store_b)
enrich_safe(store_b)
aggregate_safe(store_b)
publish_safe(store_b)
result_b = store_b.read("published")

print(f"\nResults identical across two full runs? {result_a.equals(result_b)}")

## Takeaways

| Anti-Pattern | Symptom | Fix | Airflow / Dagster Equivalent |
| :--- | :--- | :--- | :--- |
| Shared mutable state (append) | Data doubles/corrupts on retry | Overwrite output atomically | Use XCom for metadata only, write data to storage |
| Mega-task (one big function) | Full recompute on any failure | Split into checkpointed tasks | One operator per logical step |
| Non-idempotent writes | Different results on re-run | Write-to-temp-then-swap, version outputs | `depends_on_past=False` + idempotent operators |
| Tight task coupling | Cascading failures across DAG | Explicit contracts between tasks | Sensor/trigger-based dependencies |

**Golden rule:** every task should be safe to retry at any time. If re-running a task
changes the result, your pipeline has a latent bug waiting for an incident.